In [40]:
#import lib
from pandasql import sqldf

In [ ]:
#Load Dataset
df = pd.read_csv("cleaned_churn_data.csv")

In [41]:
#helper function
pysqldf = lambda q: sqldf(q, globals())

In [42]:
#Feature Engineering – Tenure Bucketing
query_tenure = """
SELECT *,
    CASE 
        WHEN tenure <= 12 THEN '0-1 Year'
        WHEN tenure > 12 AND tenure <= 24 THEN '1-2 Years'
        WHEN tenure > 24 AND tenure <= 48 THEN '2-4 Years'
        WHEN tenure > 48 AND tenure <= 60 THEN '4-5 Years'
        ELSE '5+ Years'
    END AS tenure_group
FROM df
"""

df = pysqldf(query_tenure)

df[['customerID', 'tenure', 'tenure_group']].head()

,customerID,tenure,tenure_group
0,7590-VHVEG,1,0-1 Year
1,5575-GNVDE,34,2-4 Years
2,3668-QPYBK,2,0-1 Year
3,7795-CFOCW,45,2-4 Years
4,9237-HQITU,2,0-1 Year


In [43]:
#Feature Engineering – Total Services
query_services = """
SELECT *,
    (CASE WHEN OnlineSecurity = 'Yes' THEN 1 ELSE 0 END +
     CASE WHEN OnlineBackup = 'Yes' THEN 1 ELSE 0 END +
     CASE WHEN DeviceProtection = 'Yes' THEN 1 ELSE 0 END +
     CASE WHEN TechSupport = 'Yes' THEN 1 ELSE 0 END +
     CASE WHEN StreamingTV = 'Yes' THEN 1 ELSE 0 END +
     CASE WHEN StreamingMovies = 'Yes' THEN 1 ELSE 0 END) AS total_services
FROM df
"""

df = pysqldf(query_services)
df[['customerID', 'total_services', 'Churn']].head()

,customerID,total_services,Churn
0,7590-VHVEG,1,0
1,5575-GNVDE,2,0
2,3668-QPYBK,2,1
3,7795-CFOCW,3,0
4,9237-HQITU,0,1


In [44]:
#Validation
print(df.groupby('total_services')['Churn'].mean())

total_services
0    0.214641
1    0.457557
2    0.358180
3    0.273948
4    0.223529
5    0.124780
6    0.052817
Name: Churn, dtype: float64


In [45]:
#Export Final Featured Dataset
df.to_csv('final_featured_data.csv', index=False)